In [1]:
#============================================================
# Celda 1 — Setup rutas
#============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os

# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)  # ← esto faltaba

BASE   = Path("output/evr")
SILVER = BASE / "02-silver"
GOLD   = BASE / "03-gold"
GOLD.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
# ============================================================
# Celda 2 — Inspeccionar datos de Silver
# ============================================================
df = pd.read_parquet(SILVER / "clean_evr_provincia_anio.parquet")

print(f"Shape: {df.shape}")
print(f"Años:  {sorted(df['anyo'].unique())}")
print(f"Provs: {df['provincia'].nunique()}")
print(f"Nulos:\n{df.isnull().sum()}")
print(df.head(3).to_string(index=False))

Shape: (728, 5)
Años:  [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Provs: 52
Nulos:
provincia     0
anyo          0
entradas      0
salidas       0
saldo_neto    0
dtype: int64
provincia  anyo  entradas  salidas  saldo_neto
 Albacete  2008      2023     5071       -3048
 Albacete  2009     16410     5010       11400
 Albacete  2010     11118     5176        5942


In [3]:
# ============================================================
# Celda 3 — Calcular métricas
# ============================================================
df = df.sort_values(['provincia', 'anyo']).reset_index(drop=True)

# tasa_atraccion: saldo relativo sobre entradas (evita /0)
df['tasa_atraccion'] = (
    df['saldo_neto'].astype(float) /
    df['entradas'].replace(0, np.nan).astype(float)
).round(4)

# delta_saldo_yoy: variación absoluta del saldo respecto año anterior
df['delta_saldo_yoy'] = (
    df.groupby('provincia')['saldo_neto']
    .diff()
    .astype('Int64')   # nullable int → NaN en primer año de cada provincia
)

# ranking_atraccion_anio: 1=mayor atracción neta ese año
df['ranking_atraccion_anio'] = (
    df.groupby('anyo')['saldo_neto']
    .rank(ascending=False, method='min')
    .astype(int)
)

print(f"Shape tras métricas: {df.shape}")
print(f"\nNulos:")
print(df.isnull().sum())
print(f"\nMadrid (debe tener tasa_atraccion negativa):")
print(df[df['provincia']=='Madrid'].tail(5).to_string(index=False))
print(f"\nTop 5 saldo_neto 2021:")
print(df[df['anyo']==2021].sort_values('saldo_neto', ascending=False).head(5).to_string(index=False))

Shape tras métricas: (728, 8)

Nulos:
provincia                  0
anyo                       0
entradas                   0
salidas                    0
saldo_neto                 0
tasa_atraccion             0
delta_saldo_yoy           52
ranking_atraccion_anio     0
dtype: int64

Madrid (debe tener tasa_atraccion negativa):
provincia  anyo  entradas  salidas  saldo_neto  tasa_atraccion  delta_saldo_yoy  ranking_atraccion_anio
   Madrid  2017      1639    54993      -53354        -32.5528            -4545                      52
   Madrid  2018      1771    62471      -60700        -34.2744            -7346                      52
   Madrid  2019     17324    67176      -49852         -2.8776            10848                      52
   Madrid  2020      5017    70288      -65271        -13.0100           -15419                      52
   Madrid  2021      8115    78600      -70485         -8.6858            -5214                      52

Top 5 saldo_neto 2021:
        provincia  anyo

In [4]:
# ============================================================
# Celda 4 — Guardar datos en Gold
# ============================================================
cols_gold = ['provincia','anyo','entradas','salidas','saldo_neto',
             'tasa_atraccion','delta_saldo_yoy','ranking_atraccion_anio']

gold = df[cols_gold].copy()

gold.to_parquet(GOLD / "gold_evr_provincia_anio.parquet", index=False)
gold.to_csv(    GOLD / "gold_evr_provincia_anio.csv",     index=False)

print(f"✅ Gold guardado: {gold.shape}")
print(f"\nNulos en gold:")
print(gold.isnull().sum())
print(f"\nEstadísticas:")
print(gold[['entradas','salidas','saldo_neto','tasa_atraccion']].describe().round(3))

✅ Gold guardado: (728, 8)

Nulos en gold:
provincia                  0
anyo                       0
entradas                   0
salidas                    0
saldo_neto                 0
tasa_atraccion             0
delta_saldo_yoy           52
ranking_atraccion_anio     0
dtype: int64

Estadísticas:
        entradas    salidas  saldo_neto  tasa_atraccion
count    728.000    728.000     728.000         728.000
mean    9162.110   9162.113      -0.003          -0.826
std    10927.956  10434.355   15101.766           2.781
min     1108.000   1332.000  -70485.000         -34.274
25%     3788.000   4005.500   -4084.750          -1.033
50%     6020.500   6195.000    -144.500          -0.042
75%    10279.750  10501.500    3931.000           0.444
max    76886.000  79288.000   70589.000           0.972
